In [5]:
import numpy as np
import asyncio
import pandas as pd
import chromadb
import glob
import uuid
import hashlib
import re
from collections import defaultdict
import pypdfium2 as pypdfium
import warnings
warnings.filterwarnings('ignore')

### Create a vector database and a client

In [6]:
chroma_client = chromadb.Client()

student_collection = chroma_client.create_collection(
    name="students_collection",
    embedding_function=None
)

### uploading a single PDF

In [127]:
# Load PDF from a path
pdf_path = "/Users/rezaahmadi/Downloads/Reza_Ahmadi_CV_1.pdf"

# Open the PDF
pdf = pypdfium.PdfDocument(pdf_path)

num_pages = len(pdf)

# Extract all text from the PDF as a string
all_text = ""
for page_num in range(num_pages):
    page = pdf[page_num]

    textpage = page.get_textpage()
    # Extract text from page
    text = textpage.get_text_bounded()
    all_text += text + "\n"


# Ensure correct text before moving 
# forawrd (getting rid of spaces and unicode noise)
clean_text_list = re.findall("[a-zA-Z0-9\.'@%+-]+", all_text)
clean_text = ' '.join(clean_text_list).lower()
email_list = re.findall("[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", all_text)



In [661]:
a = '''

Reza Ahmadi | Birmingham
'''
re.findall(r"[a-zA-Z0-9\.'@%+-]+", a)

['Reza', 'Ahmadi', 'Birmingham']

### Cleaning the text

In [23]:
clean_text_list = re.findall("[a-zA-Z0-9\.'@%+-]+", all_text)
clean_text = ' '.join(clean_text_list)
email = re.findall("[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", all_text)

"REZA AHMADI Birmingham UK rezaahmadi.itn@gmail.com 753-928-1647 GitHub SUMMARY Data Scientist with 2.5 years of production experience and a First Class MSc in Data Science combining strong Python skills with hands-on work across advanced predictive modelling LLM integration RAG pipelines agentic AI and MLOps practices. Experienced in building end-to-end ML systems from experimentation through to production deployment with a research mindset demonstrated through independent implementation of published academic work and production-grade AI infrastructure. Comfortable collaborating across engineering product and commercial teams to translate business goals into impactful data-driven solutions. Technical Skills Python Scikit-Learn PyTorch LLM APIs Prompt Engineering RAG Pipelines HuggingFace Transformers ChromaDB FastAPI Docker MLflow AWS SQL Pandas NumPy SciPy Statistical Modelling A B Testing Feature Engineering CI CD Git PostgreSQL Matplotlib Seaborn EXPERIENCE Snapp - Tehran Iran Juni

### Extract metadata for all the text all at once

In [286]:
from ollama import chat
from pydantic import BaseModel


class StructuredContext(BaseModel):
    name: str
    location: str
    email: str
    skills: list[str]
    total_experience: float
    job_titles: list[str]

cv_str = clean_text

response = chat(
    model='tinyllama:latest',
    messages=[{'role': 'user', 
               'content': f'''Extract the following from this CV:
    - applicant name
    - applicant location
    - applicant email
    - skills (list)
    - total years of experience (float number)
    - job titles
    
    For skills, I want you to create the format that I want. I only want the technical
    skills in short. An example of this would be : ['python', 'Git', ...] (do not blindly)
    create this but use it as an inspiration of the desired format.
    I do not want long sentences. Same goes for job titles. I want each element to be
    short and concise.

    Another important rule: everything that you will get in CVs will be in lowercase. I
    intentionally did it to make sure that the reuslts I will get are consistent 
    as I will store them in a database. So, make sure everything you return is in lowercase
    and consistent.
    
    CV:
    {cv_str}
    
    return JSON only
    '''}], format=StructuredContext.model_json_schema())

student_metadata = StructuredContext.model_validate_json(response.message.content)

### Breaking up the raw CV into chunks

In [171]:
chunks_list = clean_text.split(' ')
each_chunk_size = 30
documents_list = [' '.join(chunks_list[i:i+each_chunk_size]) for i in range(0, len(chunks_list), each_chunk_size)]

### creating a unique metadata for each student (crucial for identifying chunks that are for the same student)

In [294]:
student_metadata_id = hashlib.md5(f"{student_metadata.name}_{student_metadata.location}_{student_metadata.email}".encode()).hexdigest()

### Adding extracted metadata to the vector database

In [296]:
student_collection.add(ids=[f"{student_metadata.name}_chunk_{i}" for i in range(len(documents_list))],
               documents=documents_list,
               metadatas=[{
                   "id": student_metadata_id,
                   "name": student_metadata.name,
                   "location": student_metadata.location,
                   "skills": student_metadata.skills,
                   "experience": student_metadata.total_experience,
                   "job titles": student_metadata.job_titles,
               } for _ in range(len(documents_list))]
)

In [163]:
# for faster processing, we can use Spacy's `entity ruler` in order to find complex rule-
# based patterns that can be much faster, in cost of potentially losing some of the info

### We will create structured context for employers and query through students collection

In [216]:
jd = '''
Senior Financial AnalystWe are looking for a meticulous and strategically-minded Senior Financial Analyst to join our growing finance team. In this role, you won't just be "crunching numbers"—you’ll be translating complex data into actionable insights that drive our company’s long-term value.If you have a knack for spotting trends in a sea of spreadsheets and the communication skills to explain them to executive leadership, we want to hear from you.Key ResponsibilitiesFinancial Planning & Analysis (FP&A): Lead the annual budgeting process and quarterly forecasting cycles, ensuring accuracy and alignment with departmental goals.Performance Tracking: Develop and monitor Key Performance Indicators (KPIs) to highlight risks and opportunities within our current business model.Strategic Modeling: Build sophisticated financial models to evaluate new business ventures, capital expenditures, and potential M&A activity.Reporting: Prepare monthly and quarterly "Flash Reports" for the Board of Directors, providing clear narratives on variances between actuals and budgets.Process Improvement: Identify and implement automation opportunities within our financial reporting tools to increase efficiency and data integrity.Required Qualifications & SkillsCategoryRequirementEducationBachelor’s degree in Finance, Accounting, Economics, or related field. (MBA or CFA preferred).Experience5+ years of experience in financial analysis, investment banking, or corporate finance.Technical SkillsMastery of Excel (VBA/Macros), proficiency in SQL, and experience with BI tools (like Tableau or Power BI).Soft SkillsStrong narrative storytelling—the ability to explain the "why" behind the numbers.Why You’ll Love Working Here"Finance at our company isn't a back-office function; it’s the heartbeat of our strategy. We value transparency, intellectual curiosity, and the courage to challenge the status quo."Growth: Clear path to Finance Manager or Director levels.Flexibility: Hybrid work schedule with core "collaboration days."Impact: Your models will directly influence billion-dollar investment decisions.
'''
class JDStructuredContext(BaseModel):
    compnay_name: str
    job_title: str
    location: str
    skills_required: list[str]
    required_experience: float


response = chat(
    model='tinyllama:latest',
    messages=[{'role': 'user', 
               'content': f'''Extract the following from this CV:
    - company name
    - company location
    - skills required (list)
    - total years of required experience (float number)
    - job title
    
    For required skills, I want you to create the format that I want. I only want 
    the technical skills in short. An example of this 
    would be : ['python', 'Git', ...] (do not blindly
    create this but use it as an inspiration for the desired format).
    I do not want long sentences. Same goes for job titles. I want each element to be
    short and concise.

    Another important rule: everything that you will get in the job description will be in 
    lowercase. I intentionally did it to make sure that the reuslts I 
    will get are consistent as I will store them in a database. So, 
    make sure everything you return is in lowercase and consistent.
    
    Job Description:
    {jd}
    
    return JSON only
    '''}], format=JDStructuredContext.model_json_schema())

jd_structured_context = JDStructuredContext.model_validate_json(response.message.content)

We loop through each of the top chunks and add their distances in a list when the candidate-unique metadata id is already in the list of top applicants. This will help us get the mean of the **semantic score** for the next rounds.

In [565]:
n_top_applicants = 12
multiplier = 5

limit_results = min(student_collection.count(), n_top_applicants * multiplier)

result = student_collection.query(
    query_texts=jd,
    n_results=limit_results
)


added_applicant = 0

unique_applicants = defaultdict(lambda: {
    "dist": [],
    "meta": None,
    'final_score': 0.0
})

for each_chunk in range(limit_results):
    candidate_metadata_id = result['metadatas'][0][each_chunk]['id']
    candidate_distance = result["distances"][0][each_chunk]
    candidate_meta = result['metadatas'][0][each_chunk]

    if candidate_metadata_id not in unique_applicants.keys():
        added_applicant += 1
        if added_applicant > n_top_applicants:
            break

    unique_applicants[candidate_metadata_id]["dist"].append(candidate_distance)
    if unique_applicants[candidate_metadata_id]["meta"] is None:
        unique_applicants[candidate_metadata_id]["meta"] = candidate_meta

Next step is adding **structured score** to the pipeline to get an intelligent CV ranking system.

In [566]:
def weighted_score(skills_score, experience_score, mean_semantic_dist, weights=None):
    weights = weights or {
        'skills_weight': 0.3,
        'experience_weight': 0.2,
        'semantic_weight': 0.5
    }

    skills_score = skills_score * weights['skills_weight']
    experience_score = experience_score * weights['experience_weight']
    semantic_score = 1 / (1 + mean_semantic_dist)
    semantic_score  = semantic_score * weights['semantic_weight']

    return (skills_score + experience_score + semantic_score)


for candidate_id, candidate_data in unique_applicants.items():
    
    skills_score = (
        len(set(candidate_data['meta']['skills']) & set(jd_structured_context.skills_required)) 
        / len(jd_structured_context.skills_required)
        if jd_structured_context.skills_required else 0
    )

    experience_score = (
        min(candidate_data['meta']['experience'] / jd_structured_context.required_experience, 1.0)
        if jd_structured_context.required_experience else 0
    )

    mean_semantic_dist = np.mean(candidate_data['dist'])

    average_score = weighted_score(
        skills_score=skills_score,
        experience_score=experience_score,
        mean_semantic_dist=mean_semantic_dist,
        weights=None
    )
    
    unique_applicants[candidate_id]['final_score'] = average_score.item()

### Sort the top applicant based on their final score

In [586]:
ranked_applicants = sorted(unique_applicants.values(), key=lambda applicant: applicant['final_score'], reverse=True)

### Display applicants' info and similarity score

In [596]:
for rank, applicant in enumerate(ranked_applicants, 1):
    print(rank, applicant['meta']['name'], applicant['meta']['location'], f"{round(applicant['final_score'] * 100, 2)}%")

1 Rosa Ahmadi Binghamton, NY 29.64%


# **Experiments**

In [598]:
import pymupdf

In [ ]:
file = pymupdf.open('/Users/rezaahmadi/Downloads/Reza_Ahmadi_CV.pdf', filetype='pdf')

' '.join(page.get_text() for page in file)


'REZA AHMADI \n \nBirmingham, UK | rezaahmadi.itn@gmail.com | 753-928-1647 | GitHub  \n \nSUMMARY \nMSc Data Science graduate (First Class) with hands-on experience across the full data pipeline from feature \nengineering and selection through to model training, evaluation, and deployment. Strong mathematical and \nstatistical foundation with practical skills in Python, SQL, probabilistic modelling, NLP, LLMs, RAG, and deep \nlearning. Experienced in applying regression, classification, and clustering on real-world problems. \nComfortable working with large datasets and turning them into meaningful insight.  \n \nTechnical Skills: Python, SQL, PyTorch, Machine Learning, NumPy, Pandas, Statistics, Scikit-learn, \nXGBoost, FastAPI, Docker, Git, GitHub Actions, Hugging Face Transformers, Matplotlib, Seaborn, Plotly, \nTableau, Looker \n \n \nEXPERIENCE \nLambda BI - London, UK/Remote \nMachine Leanring Researcher - Sep 2025 – Jan 2026 \nLambda BI is a business intelligence company, specia

In [622]:
import docx2txt
import io

In [627]:
import docx

In [628]:
doc = docx.Document('/Users/rezaahmadi/Downloads/Post Saad DS ML - Variant - Copy.docx')

In [636]:
' '.join([i.text for i in doc.paragraphs])

"REZA AHMADI  Birmingham, UK | rezaahmadi.itn@gmail.com | 753-928-1647 | GitHub   SUMMARY Data Scientist with 2.5 years of production experience and a First Class MSc in Data Science, combining strong Python skills with hands-on work across advanced predictive modelling, LLM integration, RAG pipelines, and MLOps practices. Experienced in all Machine Learning paradigms, including supervised, unsupervised and reinforcement learning. Experienced in building end-to-end ML systems from experimentation through to production deployment, with a research mindset demonstrated through independent implementation of published academic work. Comfortable collaborating across engineering, product, and commercial teams to translate business goals into impactful data-driven solutions. Technical Skills: Python · SQL · Scikit-Learn · PyTorch · LLM APIs & Prompt Engineering · RAG Pipelines · HuggingFace Transformers · ChromaDB · FastAPI · Docker · MLflow · AWS · Pandas · NumPy · SciPy · Statistical Modelli

In [ ]:
a = docx2txt.process()

In [624]:
a

"REZA AHMADI\n\n\t\n\n\tBirmingham, UK | rezaahmadi.itn@gmail.com | 753-928-1647 | GitHub \n\n\t\n\n\tSUMMARY\n\nData Scientist with 2.5 years of production experience and a First Class MSc in Data Science, combining strong Python skills with hands-on work across advanced predictive modelling, LLM integration, RAG pipelines, and MLOps practices. Experienced in all Machine Learning paradigms, including supervised, unsupervised and reinforcement learning. Experienced in building end-to-end ML systems from experimentation through to production deployment, with a research mindset demonstrated through independent implementation of published academic work. Comfortable collaborating across engineering, product, and commercial teams to translate business goals into impactful data-driven solutions.\n\nTechnical Skills: Python · SQL · Scikit-Learn · PyTorch · LLM APIs & Prompt Engineering · RAG Pipelines · HuggingFace Transformers · ChromaDB · FastAPI · Docker · MLflow · AWS · Pandas · NumPy · S

In [ ]:
a = chromadb.PersistentClient(
    path='./database',
    settings=chromadb.Settings(allow_reset=True)
)


In [653]:
a = [{'name': 'ali', 'win':True}, {'name': 'mohsen', 'win': False}, {'name': 'yagoob', 'win':True}]

In [658]:
sum(1 for r in a if not r['win'])

1

In [9]:
from io import BytesIO
import docx

In [14]:
a = docx.Document('/Users/rezaahmadi/Desktop/Accountant_Job_Description.docx')
' '.join([i.text for i in a.paragraphs])

'Job Description: Accountant Job Title Accountant Location Birmingham, UK (or specify) Job Type Full-Time Salary Competitive (based on experience) Job Summary We are seeking a detail-oriented and analytical Accountant to manage financial records, ensure compliance with regulations, and support business decision-making. The ideal candidate will have strong numerical skills, integrity, and a solid understanding of accounting principles. Key Responsibilities Prepare, examine, and analyze financial statements and reports Manage accounts payable and receivable Ensure compliance with financial regulations and standards Prepare tax returns and support audits Maintain accurate financial records and documentation Assist with budgeting and forecasting Reconcile bank statements and general ledger accounts Provide financial insights to management Required Qualifications Bachelor’s degree in Accounting, Finance, or related field Professional certification (e.g., ACCA, ACA, or CIMA) preferred Proven

In [51]:
from ollama import AsyncClient
import asyncio
from pydantic import BaseModel
import time, timeit

In [74]:
attempts = 3

async def call_llm():
    await asyncio.sleep(7, result='hey')


async def retry_async_function(call_llm, *args, **kwargs):
    wait_time = 2
    for attempt in range(attempts):
        start_time = time.perf_counter()
        try:
            print('in try...')
            return await asyncio.wait_for(call_llm(*args, **kwargs), timeout=wait_time)
        
        except asyncio.TimeoutError as e:
            print(f'attempt {attempt + 1} failed')
            end_time = time.perf_counter()
            print(f"time elapsed: {end_time - start_time}")
            if attempt == attempts:
                print('error')
                raise e
            wait_time *= 2

            

In [75]:
await retry_async_function(call_llm)

in try...
attempt 1 failed
time elapsed: 2.0016793329268694
in try...
attempt 2 failed
time elapsed: 4.011765666073188
in try...
